In [1]:
! git clone https://github.com/fssdfhsdfsdk/CUDA-by-Example-source-code-for-the-book-s-examples-.git

Cloning into 'CUDA-by-Example-source-code-for-the-book-s-examples-'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 144 (delta 48), reused 48 (delta 43), pack-reused 82 (from 1)
Receiving objects: 100% (144/144), 1.42 MiB | 8.35 MiB/s, done.
Resolving deltas: 100% (70/70), done.


In [2]:
%cd /content/CUDA-by-Example-source-code-for-the-book-s-examples- 

/content/CUDA-by-Example-source-code-for-the-book-s-examples-


## add_loop_long_blocks.cu

In [9]:
%%writefile add_loop_long_blocks.cu

#include "common/book.h"

#define N (1024 * 128)



__global__ void add(int *a, int *b, int *c) {
    int tid = threadIdx.x +  blockIdx.x * blockDim.x;
    while(tid < N) {
        c[tid] = a[tid] + b[tid];
        tid += blockDim.x * gridDim.x;
    }
}


int main(void) {
    int a[N], b[N], c[N];
    int * dev_a, * dev_b, * dev_c;

    for(int i=0; i < N; i++) {
        a[i] = -i;
        b[i] = i * i;
    }

    HANDLE_ERROR( cudaMalloc((void **)&dev_a, N * sizeof(int)));
    HANDLE_ERROR( cudaMalloc((void **)&dev_b, N * sizeof(int)));
    HANDLE_ERROR( cudaMalloc((void **)&dev_c, N * sizeof(int)));

    HANDLE_ERROR( cudaMemcpy(dev_a, a, N * sizeof(int), cudaMemcpyHostToDevice));
    HANDLE_ERROR( cudaMemcpy(dev_b, b, N * sizeof(int), cudaMemcpyHostToDevice));

    add<<<128,128>>>(dev_a, dev_b, dev_c);

    HANDLE_ERROR( cudaMemcpy(c, dev_c, N * sizeof(int), cudaMemcpyDeviceToHost));

    for(int i=0; i < N; i++) {
        if (a[i] + b[i] != c[i]) {
            printf("Error: %d + %d != %d \n", a[i], b[i], c[i]);
        }
    }



    cudaFree(dev_a);
    cudaFree(dev_b);
    cudaFree(dev_c);

    return 0;
}

Overwriting add_loop_long_blocks.cu


In [10]:
! nvcc add_loop_long_blocks.cu -arch=sm_75 -o hello && date; ./hello; date

Mon Mar  9 02:58:33 AM UTC 2026
Mon Mar  9 02:58:34 AM UTC 2026
